
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

## 03 - Delta Lake 기능을 사용하여 Databricks SQL

이번 수업에서는 Delta Lakes가 Databricks SQL 테이블 작업을 위해 제공하는 기능들을 살펴보겠습니다. 이를 위해 표를 만들고 그것을 조작한 후 log에서 결과를 확인하는 과정을 살펴보겠습니다. 

`db_catalog` 카탈로그에서 `db_schema`가 필요합니다.

### 실행 설정 (필수)
아래 셀의 재생 아이콘을 사용해 이 시연을 실행해 보세요. 이는 테이블과 뷰를 만들 스키마를 초기화하는 데 필요합니다. 스키마 이름의 메모를 가져가세요. 나중에 활동할 때 그것이 필요하기 때문입니다.

In [0]:
%sql
USE CATALOG db_catalog;
USE SCHEMA db_schema;

### 1부: 테이블에 데이터 생성 및 추가
##### A. 카탈로그 및 스키마 설정 및 조회

In [0]:
-- 카탈로그 및 스키마 설정
USE CATALOG db_catalog;
USE SCHEMA db_schema;

-- 현재 카탈로그와 스키마 보기
SELECT current_catalog() AS catalog_name, current_schema() AS schema_name

#### B. 테이블 만들기
ID 필드는 테이블 **delta_students**에서 **BIGINT** 타입이며, 그 값은 단조 증가 숫자로 자동으로 생성됩니다. 그것은 테이블의 기본 키로 간주할 수 있는데, **INSERT** 중에는 이 값을 전달하지 않습니다.

In [0]:
CREATE OR REPLACE TABLE delta_students (
  id BIGINT GENERATED ALWAYS AS IDENTITY,
  name STRING,
  grade FLOAT,
  country STRING
);

##### C. 레코드 삽입 및 업데이트

In [0]:
INSERT INTO
  delta_students (name, grade, country)
VALUES
  ('Lucas', 4.0, "Italy"),
  ("Ana", 3.5, "Germany");

SELECT
  *
FROM
  delta_students;

그 다음, 표에 추가 기록을 추가할 예정입니다.

In [0]:
INSERT INTO
  delta_students (name, grade, country)
VALUES
  ('Mary', 2.0, "Finland"),
  ("John", 3.5, "USA"),
  ("Judith", 4.0, "Singapore");

다음으로, 데이터에 몇 가지 **업데이트**를 할 예정입니다. 한 학생의 국가를 바꾸고 다른 학생의 성적을 수정할 것입니다. 마지막으로, 현재 데이터 세트를 그것을 현재 상태로 뷰할 것입니다.

In [0]:
UPDATE
  delta_students
SET
  country = "France"
WHERE
  name = 'Lucas';

UPDATE
  delta_students
SET
  grade = 2.5
WHERE
  id = 3;

SELECT
  *
FROM
  delta_students
ORDER BY
  id;

### 2부: 데이터 분석
**DESCRIBE HISTORY**와 **VERSION AS OF**를 사용해 그것을 탐색하며 표의 메타데이터를 좀 더 자세히 살펴보겠습니다.

##### A. 테이블 기록 보기

In [0]:
DESCRIBE HISTORY delta_students;

##### B. 이전 버전 탐색

이력에서 보시다시피, 우리가 한 모든 변경과 삽입은 Delta가 자동으로 실행한 최적화와 함께 로그됩니다. **VERSION AS OF**를 사용해 이력 테이블에서 해당 버전에서 데이터가 어떻게 보였는지 뷰할 수 있습니다. 

그럼 **버전 1** 시점에서 첫 두 레코드를 삽입한 표를 살펴보겠습니다.

In [0]:
SELECT * FROM delta_students VERSION AS OF 1 ORDER BY id;

그럼 **버전 2**를 살펴보세요.

또는 `@v2` 대신 `VERSION AS OF 2`

In [0]:
SELECT * FROM delta_students@v2 ORDER BY id;

##### C. 변화 분석
테이블에서 수행되는 모든 연산은 새로운 버전을 만듭니다. 두 버전을 비교해 어떤 변경이 이루어졌는지 더 잘 이해해 봅시다.

In [0]:
SELECT * FROM delta_students@v2
MINUS
SELECT * FROM delta_students@v1

### 3부: 테이블 복원

만약 테이블에 무슨 문제가 생겼다면, 아마도 동료가 실수로 테이블의 모든 기록을 삭제했을 수도 있습니다.

In [0]:
DELETE FROM delta_students;

In [0]:
SELECT * FROM delta_students;

그래서 이제 테이블에 따라 다른 보고서를 생성하기 전에 문제를 해결해야 합니다. 따라서 **DESCRIBE HISTORY**를 사용하면 **DELETE** 쿼리이 실행되기 전에 존재했던 버전을 찾을 수 있습니다.

In [0]:
DESCRIBE HISTORY delta_students;

버전 목록에 따르면 마지막 업데이트는 버전 4였으므로, 버전 4를 사용해 **RESTORE TABLE** 명령어를 사용해 테이블을 이전 버전으로 복원할 예정입니다.

In [0]:
RESTORE TABLE delta_students VERSION AS OF 4;

그래서 이제 표를 확인해보면, 동료가 실수로 삭제하기 전의 존재하던 모습으로 돌아왔어요.

In [0]:
SELECT * FROM delta_students ORDER BY id;

---
#### 결론

보시다시피, Delta Lake는 테이블을 사용할 때 여러 가지 이점을 제공합니다. 예를 들어, 테이블을 이전 상태로 되돌릴 수 있는 기능이 있습니다.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>